# Notebook for retrieving Open-Meteo Weather Data


In [ ]:
#imports
import requests
import json
import csv
import time


List of cities:

first run is major US cities + some from North America

In [ ]:

# ── City list ──────────────────────────────────────────────────────────────────
CITIES = [
    # USA
    "New York City",
    "Chicago",
    "Los Angeles",
    "Washington",
    "Dallas",
    "San Francisco",
    "Miami",
    "Seattle",
    "Boston",
    "Denver",
    "Houston",
    "Phoenix",
    "Portland",
    "Philadelphia",
    "Honolulu",
    "Salt Lake City",
    "Nashville",
    "Oklahoma City",
    "Las Vegas",
    "Baltimore",
    "Kansas City",
    "Omaha",
    "Minneapolis",
    "Cleveland",
    # Additional North American cities
    "Toronto",
    "Ottawa",
    "Halifax",
    "Mexico City",
    "Havana",
    "Montreal",
    "Tijuana",
    "Panama City",
]

In [ ]:

BASE_URL = "https://geocoding-api.open-meteo.com/v1/search"
FILEPATH_OUT= "location-list/" 

# ── Fetch geodata for a single city ───────────────────────────────────────────
def fetch_geodata(city_name: str, count: int = 1) -> dict | None:
    """
    Query the Open-Meteo geocoding API for `city_name`.
    Returns the top result as a flat dict, or None if nothing found.
    """
    params = {
        "name": city_name,
        "count": count,
        "language": "en",
        "format": "json",
    }
    response = requests.get(BASE_URL, params=params, timeout=10)
    response.raise_for_status()
    data = response.json()

    results = data.get("results")
    if not results:
        print(f"  ⚠  No results for: {city_name}")
        return None

    top = results[0]
    return {
        "query":        city_name,
        "id":           top.get("id"),
        "name":         top.get("name"),
        "latitude":     top.get("latitude"),
        "longitude":    top.get("longitude"),
        "elevation":    top.get("elevation"),
        "country_code": top.get("country_code"),
        "country":      top.get("country"),
        "admin1":       top.get("admin1"),       # state / province
        "admin2":       top.get("admin2"),
        "timezone":     top.get("timezone"),
        "population":   top.get("population"),
    }


# ── Main routine ───────────────────────────────────────────────────────────────
def fetch_all_cities(
    cities: list[str],
    json_out: str = "geodata.json",
    csv_out:  str = "geodata.csv",
    delay:    float = 0.2,          # seconds between requests (be polite)
) -> list[dict]:
    """
    Iterate over `cities`, fetch geodata for each, then save to JSON and CSV.

    Parameters
    ----------
    cities   : list of city name strings
    json_out : output path for the JSON file
    csv_out  : output path for the CSV file
    delay    : pause between API requests (seconds)

    Returns
    -------
    List of result dicts (one per city that returned a hit).
    """
    results = []

    for city in cities:
        print(f"Fetching: {city} …")
        try:
            record = fetch_geodata(city)
            if record:
                results.append(record)
        except requests.RequestException as exc:
            print(f"  ✗ Request error for '{city}': {exc}")
        time.sleep(delay)

    # ── Save JSON ──────────────────────────────────────────────────────────────
    with open(json_out, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2, ensure_ascii=False)
    print(f"\n✓ JSON saved → {json_out}  ({len(results)} records)")

    # ── Save CSV ───────────────────────────────────────────────────────────────
    if results:
        fieldnames = list(results[0].keys())
        with open(csv_out, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(results)
        print(f"✓ CSV  saved → {csv_out}")

    return results


# ── Entry point ────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    geodata = fetch_all_cities(
        cities=CITIES,
        json_out=f"{FILEPATH_OUT}geodata.json",  # ← f-string interpolates the variable
        csv_out=f"{FILEPATH_OUT}geodata.csv",    # ← f-string interpolates the variable
    )
    print(f"\nDone. {len(geodata)} cities geocoded.")    

Use the list generated in previous steps to get historical weather data from https://open-meteo.com/en/docs/historical-weather-api

In [ ]:
"""
fetch_weather_history.py
────────────────────────
Fetch daily historical weather (2000-01-01 → today) from the Open-Meteo
Historical Weather API for every city in the geocoding JSON produced by
geocode_cities.py.

Dependencies (install once):
    pip install openmeteo-requests requests-cache retry-requests pandas

Usage (in a notebook):
    # Set LOCATION_LIST_FILEPATH below, then run:
    %run fetch_weather_history.py
    # OR call directly:
    df = fetch_all_weather()
"""

import json
import time
from datetime import date, datetime
from pathlib import Path

import pandas as pd
import requests_cache
import openmeteo_requests
from retry_requests import retry

In [ ]:
#need to put cities list into 5 city chunks (5,000 calls per hour, 10,000 calls per day)
cities_1 = CITIES[0:5]
cities_2 = CITIES[5:10]
cities_3 = CITIES[10:15]
cities_4 = CITIES[15:20]
#can get up to cities_4 in by tomorrow (May 4th) probably

cities_5 = CITIES[20:25]
cities_6 = CITIES[25:30]
cities_7 = CITIES[30:35]
cities_8 = CITIES[35:40]




In [44]:
"""
fetch_weather_history.py
────────────────────────
Fetch daily historical weather (2000-01-01 → today) from the Open-Meteo
Historical Weather API for every city in the geocoding JSON produced by
geocode_cities.py.

Dependencies (install once):
    pip install openmeteo-requests requests-cache retry-requests pandas

Usage (in a notebook):
    # Set LOCATION_LIST_FILEPATH and CHUNK_NUMBER below, then run:
    %run fetch_weather_history.py
    # OR call directly:
    df = fetch_all_weather()
"""

import json
import time
from datetime import date
from pathlib import Path

import pandas as pd
import requests_cache
import openmeteo_requests
from retry_requests import retry

# ══════════════════════════════════════════════════════════════════════════════
# ── USER SETTINGS  ────────────────────────────────────────────────────────────
# ══════════════════════════════════════════════════════════════════════════════

# Path to the geodata JSON produced by geocode_cities.py
LOCATION_LIST_FILEPATH = "location-list/geodata.json"

# ── Chunking ───────────────────────────────────────────────────────────────────
# The full city list is split into chunks of CHUNK_SIZE.
# Set CHUNK_NUMBER to the chunk you want to run (1-indexed).
# e.g. with 32 cities and CHUNK_SIZE=5:
#   Chunk 1 → cities 1–5
#   Chunk 2 → cities 6–10
#   ...
#   Chunk 7 → cities 31–32
# Each chunk's output is saved separately, e.g. weather_history_chunk1.csv
# Run combine_chunks() at the end to merge all chunks into one file.


CHUNK_NUMBER = 1   # ← UPDATE THIS before each run
CHUNK_SIZE   = 5   # ← cities per chunk (keep at 5 to stay within API limits)

# Date range
START_DATE = "2000-01-01"
END_DATE   = date.today().isoformat()

# Output paths — chunk number is appended automatically
OUTPUT_DIR  = "weather-data"
OUTPUT_CSV  = f"{OUTPUT_DIR}/weather_history_chunk{CHUNK_NUMBER}.csv"
OUTPUT_JSON = f"{OUTPUT_DIR}/weather_history_chunk{CHUNK_NUMBER}.json"

# Final merged output (used by combine_chunks())
MERGED_CSV  = f"{OUTPUT_DIR}/weather_history.csv"
MERGED_JSON = f"{OUTPUT_DIR}/weather_history.json"

# How many cities to batch in a single API call
# With 14 variables x ~9,300 days, keep this at 1
BATCH_SIZE  = 1

# Seconds to pause between cities (61 = safe within per-minute limit)
BATCH_DELAY = 61

# ══════════════════════════════════════════════════════════════════════════════
# ── WEATHER VARIABLES  ────────────────────────────────────────────────────────
# ══════════════════════════════════════════════════════════════════════════════

DAILY_VARIABLES = [
    "weather_code",                 # Weather Code (WMO)
    "temperature_2m_mean",          # Mean Temperature (2 m)
    "apparent_temperature_mean",    # Mean Apparent Temperature
    "rain_sum",                     # Rain Sum
    "precipitation_hours",          # Precipitation Hours
    "showers_sum",                  # Showers Sum
    "snowfall_sum",                 # Snowfall Sum
    "daylight_duration",            # Daylight Duration (s)
    "sunshine_duration",            # Sunshine Duration (s)
    "wind_gusts_10m_max",           # Maximum Wind Gusts
    "wind_speed_10m_mean",          # Mean Wind Speed
    "wind_gusts_10m_mean",          # Mean Wind Gusts
    "dew_point_2m_mean",            # Mean Dewpoint
    "cloud_cover_mean",             # Mean Cloud Cover
]

COL_RENAME = {
    "weather_code":              "weather_code",
    "temperature_2m_mean":       "temp_mean_c",
    "apparent_temperature_mean": "apparent_temp_mean_c",
    "rain_sum":                  "rain_sum_mm",
    "precipitation_hours":       "precipitation_hours",
    "showers_sum":               "showers_sum_mm",
    "snowfall_sum":              "snowfall_sum_cm",
    "daylight_duration":         "daylight_duration_s",
    "sunshine_duration":         "sunshine_duration_s",
    "wind_gusts_10m_max":        "wind_gusts_max_kmh",
    "wind_speed_10m_mean":       "wind_speed_mean_kmh",
    "wind_gusts_10m_mean":       "wind_gusts_mean_kmh",
    "dew_point_2m_mean":         "dewpoint_mean_c",
    "cloud_cover_mean":          "cloud_cover_mean_pct",
}

# ══════════════════════════════════════════════════════════════════════════════
# ── API CLIENT SETUP  ─────────────────────────────────────────────────────────
# ══════════════════════════════════════════════════════════════════════════════

def _make_client() -> openmeteo_requests.Client:
    cache_session = requests_cache.CachedSession(".om_cache", expire_after=-1)
    retry_session = retry(cache_session, retries=5, backoff_factor=0.4)
    return openmeteo_requests.Client(session=retry_session)

# ══════════════════════════════════════════════════════════════════════════════
# ── CORE FETCH LOGIC  ─────────────────────────────────────────────────────────
# ══════════════════════════════════════════════════════════════════════════════

def _fetch_batch(
    client: openmeteo_requests.Client,
    batch: list[dict],
    start_date: str,
    end_date: str,
) -> list[pd.DataFrame]:
    lats = [c["latitude"]  for c in batch]
    lons = [c["longitude"] for c in batch]

    params = {
        "latitude":        lats,
        "longitude":       lons,
        "start_date":      start_date,
        "end_date":        end_date,
        "daily":           DAILY_VARIABLES,
        "timezone":        "auto",
        "wind_speed_unit": "kmh",
    }

    url = "https://archive-api.open-meteo.com/v1/archive"
    responses = client.weather_api(url, params=params)

    frames = []
    for city_meta, resp in zip(batch, responses):
        daily = resp.Daily()

        dates = pd.date_range(
            start     = pd.to_datetime(daily.Time(),    unit="s", utc=True),
            end       = pd.to_datetime(daily.TimeEnd(), unit="s", utc=True),
            freq      = pd.Timedelta(seconds=daily.Interval()),
            inclusive = "left",
        ).date

        data = {"date": dates}
        for i, var_name in enumerate(DAILY_VARIABLES):
            data[var_name] = daily.Variables(i).ValuesAsNumpy()

        df = pd.DataFrame(data)
        df.rename(columns=COL_RENAME, inplace=True)

        df.insert(1, "city",      city_meta.get("query",    city_meta.get("name", "")))
        df.insert(2, "name",      city_meta.get("name",     ""))
        df.insert(3, "country",   city_meta.get("country",  ""))
        df.insert(4, "admin1",    city_meta.get("admin1",   ""))
        df.insert(5, "latitude",  city_meta.get("latitude",  resp.Latitude()))
        df.insert(6, "longitude", city_meta.get("longitude", resp.Longitude()))
        df.insert(7, "timezone",  resp.Timezone().decode() if isinstance(resp.Timezone(), bytes) else resp.Timezone())

        frames.append(df)

    return frames

# ══════════════════════════════════════════════════════════════════════════════
# ── PUBLIC ENTRY POINT  ───────────────────────────────────────────────────────
# ══════════════════════════════════════════════════════════════════════════════

def fetch_all_weather(
    location_filepath: str = LOCATION_LIST_FILEPATH,
    chunk_number: int      = CHUNK_NUMBER,
    chunk_size: int        = CHUNK_SIZE,
    start_date: str        = START_DATE,
    end_date: str          = END_DATE,
    output_csv: str        = OUTPUT_CSV,
    output_json: str       = OUTPUT_JSON,
    batch_size: int        = BATCH_SIZE,
    batch_delay: float     = BATCH_DELAY,
) -> pd.DataFrame:
    # ── Load city list ─────────────────────────────────────────────────────────
    geo_path = Path(location_filepath)
    if not geo_path.exists():
        raise FileNotFoundError(
            f"Could not find geodata file at: {geo_path}\n"
            "Set LOCATION_LIST_FILEPATH to the path of your geodata.json."
        )

    with geo_path.open("r", encoding="utf-8") as f:
        all_cities = json.load(f)

    # ── Slice the chunk ────────────────────────────────────────────────────────
    total_chunks = (len(all_cities) + chunk_size - 1) // chunk_size
    if chunk_number < 1 or chunk_number > total_chunks:
        raise ValueError(f"CHUNK_NUMBER must be between 1 and {total_chunks} (got {chunk_number})")

    start_idx = (chunk_number - 1) * chunk_size
    end_idx   = start_idx + chunk_size
    cities    = all_cities[start_idx:end_idx]

    print(f"Total cities  : {len(all_cities)}")
    print(f"Chunk         : {chunk_number}/{total_chunks}  (cities {start_idx+1}–{min(end_idx, len(all_cities))})")
    print(f"Cities in run : {[c.get('query', c.get('name')) for c in cities]}")
    print(f"Date range    : {start_date} → {end_date}")
    print(f"Variables     : {len(DAILY_VARIABLES)}")
    print(f"Est. run time : ~{(len(cities) - 1) * batch_delay / 60:.1f} min\n")

    # Ensure output directory exists
    Path(output_csv).parent.mkdir(parents=True, exist_ok=True)

    client     = _make_client()
    all_frames = []

    total_batches = (len(cities) + batch_size - 1) // batch_size
    for batch_num, i in enumerate(range(0, len(cities), batch_size), start=1):
        batch      = cities[i : i + batch_size]
        city_names = [c.get("query", c.get("name", "?")) for c in batch]
        print(f"Batch {batch_num}/{total_batches}: {city_names}")

        try:
            frames = _fetch_batch(client, batch, start_date, end_date)
            all_frames.extend(frames)
            print(f"  ✓  {sum(len(f) for f in frames):,} rows fetched")
        except Exception as exc:
            print(f"  ✗  Error fetching batch {batch_num}: {exc}")

        if batch_num < total_batches:
            print(f"  ⏳ Waiting {batch_delay}s before next city...")
            time.sleep(batch_delay)

    if not all_frames:
        print("No data fetched.")
        return pd.DataFrame()

    combined = pd.concat(all_frames, ignore_index=True)
    combined.sort_values(["city", "date"], inplace=True)
    combined.reset_index(drop=True, inplace=True)

    print(f"\n✓ Total rows: {len(combined):,}")

    combined.to_csv(output_csv, index=False)
    print(f"✓ CSV  saved → {output_csv}")

    json_ready = combined.copy()
    json_ready["date"] = json_ready["date"].astype(str)
    json_ready.to_json(output_json, orient="records", indent=2, force_ascii=False)
    print(f"✓ JSON saved → {output_json}")

    return combined

# ══════════════════════════════════════════════════════════════════════════════
# ── COMBINE ALL CHUNKS  ───────────────────────────────────────────────────────
# ══════════════════════════════════════════════════════════════════════════════

def combine_chunks(
    output_dir: str        = OUTPUT_DIR,
    merged_csv: str        = MERGED_CSV,
    merged_json: str       = MERGED_JSON,
    chunk_size: int        = CHUNK_SIZE,
    location_filepath: str = LOCATION_LIST_FILEPATH,
) -> pd.DataFrame:
    """
    Merge all per-chunk CSVs into a single weather_history.csv / .json.
    Run this after all chunks are complete.

    Usage:
        from fetch_weather_history import combine_chunks
        df = combine_chunks()
    """
    with open(location_filepath, "r") as f:
        all_cities = json.load(f)
    total_chunks = (len(all_cities) + chunk_size - 1) // chunk_size

    print(f"Looking for {total_chunks} chunk files in '{output_dir}/'...\n")

    frames = []
    for n in range(1, total_chunks + 1):
        path = Path(output_dir) / f"weather_history_chunk{n}.csv"
        if path.exists():
            df = pd.read_csv(path)
            frames.append(df)
            print(f"  ✓ Chunk {n}: {len(df):,} rows")
        else:
            print(f"  ⚠ Chunk {n} not found: {path}  (skipping)")

    if not frames:
        print("No chunk files found.")
        return pd.DataFrame()

    combined = pd.concat(frames, ignore_index=True)
    combined.sort_values(["city", "date"], inplace=True)
    combined.reset_index(drop=True, inplace=True)

    combined.to_csv(merged_csv, index=False)
    print(f"\n✓ Merged CSV  → {merged_csv}  ({len(combined):,} total rows)")

    json_ready = combined.copy()
    json_ready["date"] = json_ready["date"].astype(str)
    json_ready.to_json(merged_json, orient="records", indent=2, force_ascii=False)
    print(f"✓ Merged JSON → {merged_json}")

    return combined

# ══════════════════════════════════════════════════════════════════════════════
# ── CLI / notebook entry point  ───────────────────────────────────────────────
# ══════════════════════════════════════════════════════════════════════════════

if __name__ == "__main__":
    df = fetch_all_weather()
    if not df.empty:
        print("\nSample (first 5 rows):")
        print(df.head().to_string(index=False))

Total cities  : 32
Chunk         : 1/7  (cities 1–5)
Cities in run : ['New York City', 'Chicago', 'Los Angeles', 'Washington', 'San Francisco']
Date range    : 2000-01-01 → 2026-05-03
Variables     : 14
Est. run time : ~4.1 min

Batch 1/5: ['New York City']
  ✓  9,620 rows fetched
  ⏳ Waiting 61s before next city...
Batch 2/5: ['Chicago']
  ✓  9,620 rows fetched
  ⏳ Waiting 61s before next city...
Batch 3/5: ['Los Angeles']
  ✗  Error fetching batch 3: failed to request 'https://archive-api.open-meteo.com/v1/archive': {'reason': 'Daily API request limit exceeded. Please try again tomorrow.', 'error': True}
  ⏳ Waiting 61s before next city...
Batch 4/5: ['Washington']
  ✗  Error fetching batch 4: failed to request 'https://archive-api.open-meteo.com/v1/archive': {'error': True, 'reason': 'Daily API request limit exceeded. Please try again tomorrow.'}
  ⏳ Waiting 61s before next city...
Batch 5/5: ['San Francisco']
  ✗  Error fetching batch 5: failed to request 'https://archive-api.open-

# FOR TOMORROW MAY 4:


- Run Chunks 2 and 3
- add to normalization
- add to website

Look at cached data

In [ ]:
import openmeteo_requests
import pandas as pd

# Re-make the client (same as the main script)
import requests_cache
from retry_requests import retry

cache_session = requests_cache.CachedSession(".om_cache", expire_after=-1)
retry_session = retry(cache_session, retries=5, backoff_factor=0.4)
openmeteo     = openmeteo_requests.Client(session=retry_session)

# Re-request the same URL — will be served from cache instantly
url = "https://archive-api.open-meteo.com/v1/archive"
params = {
    "latitude":       40.71427,
    "longitude":      -74.00597,
    "start_date":     "2000-01-01",
    "end_date":       "2026-05-03",
    "daily":          ["weather_code","temperature_2m_mean","apparent_temperature_mean",
                       "precipitation_sum","precipitation_hours","showers_sum",
                       "snowfall_sum","daylight_duration","sunshine_duration",
                       "uv_index_max","wind_gusts_10m_max","wind_speed_10m_mean",
                       "wind_gusts_10m_mean","dew_point_2m_mean","visibility_mean"],
    "timezone":       "auto",
    "wind_speed_unit":"kmh",
}

responses = openmeteo.weather_api(url, params=params)
response  = responses[0]

daily = response.Daily()
dates = pd.date_range(
    start     = pd.to_datetime(daily.Time(),    unit="s", utc=True),
    end       = pd.to_datetime(daily.TimeEnd(), unit="s", utc=True),
    freq      = pd.Timedelta(seconds=daily.Interval()),
    inclusive = "left",
).date

DAILY_VARIABLES = ["weather_code","temperature_2m_mean","apparent_temperature_mean",
                   "precipitation_sum","precipitation_hours","showers_sum",
                   "snowfall_sum","daylight_duration","sunshine_duration",
                   "uv_index_max","wind_gusts_10m_max","wind_speed_10m_mean",
                   "wind_gusts_10m_mean","dew_point_2m_mean","visibility_mean"]

data = {"date": dates}
for i, var in enumerate(DAILY_VARIABLES):
    data[var] = daily.Variables(i).ValuesAsNumpy()

df = pd.DataFrame(data)
print(df.head(10))

In [ ]:
df.head()